In [2]:
from openai import OpenAI
import json
from dotenv import load_dotenv
import os
load_dotenv(".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [11]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal, Union

class Replacement(BaseModel):
    original: str = Field(..., description="The base text from first PDF")
    text: str = Field(..., description="The new text that place in")
    type:  Literal["replace"]
    
class Addition(BaseModel):
    text: str = Field(..., description="The text that add in")
    type: Literal["add"]
    
class Deletion(BaseModel):
    text: str = Field(..., description="The text that delete from")
    type:  Literal["delete"]
    
class Page(BaseModel):
    page_number: int
    modify_detection: Optional[List[Union['Replacement', 'Addition', 'Deletion']]] 

class Detection_Result(BaseModel):
    page: List[Page]
    modified: Literal["yes", "no"] = Field(..., description="Yes if detect any different, no if not")
    

# Export to JSON Schema
print(Detection_Result.schema_json(indent=2))

{
  "$defs": {
    "Addition": {
      "properties": {
        "text": {
          "description": "The text that add in",
          "title": "Text",
          "type": "string"
        },
        "type": {
          "const": "add",
          "title": "Type",
          "type": "string"
        }
      },
      "required": [
        "text",
        "type"
      ],
      "title": "Addition",
      "type": "object"
    },
    "Deletion": {
      "properties": {
        "text": {
          "description": "The text that delete from",
          "title": "Text",
          "type": "string"
        },
        "type": {
          "const": "delete",
          "title": "Type",
          "type": "string"
        }
      },
      "required": [
        "text",
        "type"
      ],
      "title": "Deletion",
      "type": "object"
    },
    "Page": {
      "properties": {
        "page_number": {
          "title": "Page Number",
          "type": "integer"
        },
        "modify_detection": {
 

/tmp/ipykernel_26645/1647056648.py:27: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  print(Detection_Result.schema_json(indent=2))


In [ ]:
from openai import OpenAI
client = OpenAI()

file_1 = client.files.create(
    file=open("data/Invoice-900F4466-0001.pdf", "rb"),
    purpose="user_data"
)

file_2 = client.files.create(
    file=open("data/Invoice-900F4466-0001 - Copy (1).pdf", "rb"),
    purpose="user_data"
)

file_3 = client.files.create(
    file=open("data/Invoice-900F4466-0001 - Copy (2).pdf", "rb"),
    purpose="user_data"
)


{'page': [{'page_number': 1,
   'modify_detection': [{'type': 'replace',
     'text': 'Invoice number 900F4466-0001 (base) replaced with Invoice number901F44660001 (customer)'},
    {'type': 'replace',
     'text': 'Date of issue April 9, 2025 (base) replaced with April 5, 2025 (customer)'},
    {'type': 'replace',
     'text': 'Date due April 9, 2025 (base) replaced with April 5, 2025 (customer)'},
    {'type': 'replace',
     'text': '$2.00 USD due April 9, 2025 (base) replaced with $3.00 USD due April 9,2025 (customer)'},
    {'type': 'replace',
     'text': 'Pre-pay for inference 1 $2.00 $2.00 (base) replaced with Pre-pay for inference 1 $3.00 $3.00 (customer)'},
    {'type': 'replace',
     'text': 'Subtotal $2.00 (base) replaced with Subtotal $3.00 (customer)'},
    {'type': 'replace',
     'text': 'Total $2.00 (base) replaced with Total $3.00 (customer)'},
    {'type': 'replace',
     'text': 'Amount due $2.00 USD (base) replaced with Amount due $3.00 USD (customer)'},
    {'typ

In [31]:
response = client.responses.create(
    model="gpt-4o",
    input=[
                {
                "role": "system",
                "content": [
                    {
                        "type": "input_text",
                        "text": """
        **Role**: You are a good PDF comparer:

        **Detailed Mission**:
        - You will receive input text about 2 PDF - all extracted text, in each page.
        - These 2 PDF are expected to be the same, but the second one is received from customer, so they could change some thing - what you will detect. 
        - You need to detect any different in 3 type: replace, add, delete if exist in each page, if no, leave None. Please follow the given structured to answer.

        **Attention**:
        - Because the second PDF is scanned, so the overal position may change a bit - lead to Text Extract may not in order from the first, but this should be just a slight change, so do not strict much about the position
        - You should care about meaning, sentence level

        """
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "file_id": file_1.id,
                    },
                    {
                        "type": "input_text",
                        "text": "This is the base PDF from us",
                    },
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "file_id": file_3.id,
                    },
                    {
                        "type": "input_text",
                        "text": "This is PDF received back from customer",
                    },
                ]
            },
            
    ],
    text = {
    "format": {
        "type": "json_schema",
        "name": "detection_result",
        "schema": {
            "type": "object",
            "properties": {
                "page": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "page_number": {"type": "integer"},
                            "modify_detection": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "type": {
                                            "type": "string",
                                            "enum": ["replace", "add", "delete"]
                                        },
                                        "text": {
                                            "type": "string",
                                            "description": "The new text (for 'replace' and 'add'), or text that was deleted (for 'delete'), just return the replaced / add / delete text only from the second pdf"
                                        },
                                        "init": {
                                            "type": "string",
                                            "description": "For replace, return the base text from the first pdf, if add and delete, return empty string "
                                        }
                                    },
                                    "required": ["type", "text", "init"],
                                    "additionalProperties": False
                                }
                            }
                        },
                        "required": ["page_number", "modify_detection"],
                        "additionalProperties": False
                    }
                },
                "modified": {
                    "type": "string",
                    "enum": ["yes", "no"],
                    "description": "Yes if detect any different, no if not"
                }
            },
            "required": ["page", "modified"],
            "additionalProperties": False
        },
        "strict": True
    }
}

)

event = json.loads(response.output_text)
event

{'page': [{'page_number': 1,
   'modify_detection': [{'type': 'replace',
     'text': '$3.00 USD due April 9, 2025',
     'init': '$2.00 USD due April 9, 2025'},
    {'type': 'replace', 'text': '$3.00', 'init': '$2.00'},
    {'type': 'replace', 'text': '901F44660001', 'init': '900F4466-0001'},
    {'type': 'replace', 'text': 'April 5, 2025', 'init': 'April 9, 2025'},
    {'type': 'replace', 'text': '121000148', 'init': '121000248'}]}],
 'modified': 'yes'}

In [5]:
import requests

# Define the endpoint URL
url = "http://localhost:8006/upload-pdf"  # Change port if different

# Paths to your two local PDF files
pdf_path_1 = "data/Invoice-900F4466-0001.pdf"
pdf_path_2 = "data/Invoice-900F4466-0001.pdf"

# Prepare the files to send as multipart/form-data
files = {
    "file_1": ("Invoice-900F4466-0001.pdf", open(pdf_path_1, "rb"), "application/pdf"),
    "file_2": ("Invoice-900F4466-0001.pdf", open(pdf_path_2, "rb"), "application/pdf"),
}

# Send the POST request
response = requests.post(url, files=files)

# Print the response from the server
print("Status code:", response.status_code)
print("Response JSON:", response.json())

Status code: 200
Response JSON: {'status': 'success', 'mistral_response': '{\n  "page": [\n    {\n      "page_number": 1,\n      "modify_detection": null\n    }\n  ],\n  "modified": "no"\n}'}
